[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_05_classical_ml/ch23_classification_calibration/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 23 Walkthrough: Classification Foundations — Logistic Regression, Calibration, ROC/PR Curves & Class Imbalance

**Book Reference:** Chapter 23, Sections 23.1 – 23.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every clinical classification experiment uses the bundled `mch_survey_sample` maternal and child health dataset.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import warnings
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
wealth_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
mch["wealth_num"] = mch["wealth_quintile"].map(wealth_map)
feat_cols = [
    "maternal_age_years", "anc_visits", "wealth_num", "is_urban"
]
clean = mch[["maternal_anemia", "child_stunted"] + feat_cols].dropna()
print(f"Raw shape: {mch_raw.shape} | Complete rows: {len(clean)}")
print(
    f"Anemia rate: {clean['maternal_anemia'].mean():.4f} | "
    f"Stunting rate: {clean['child_stunted'].mean():.4f}"
)


## Section 23.3.1 — Sigmoid & Binary Cross-Entropy (Log-Loss) from Scratch

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500.0, 500.0)))

def bce_loss(y_true, y_prob, eps=1e-15):
    y = np.asarray(y_true, dtype=float)
    p = np.clip(np.asarray(y_prob, dtype=float), eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

# Standardized socio-demographic features -> maternal_anemia
X_raw = clean[["anc_visits", "wealth_num"]].to_numpy(dtype=float)
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
y_anem = clean["maternal_anemia"].to_numpy(dtype=float)

w = np.zeros(X_std.shape[1], dtype=float)
b = 0.0
losses = []
for _ in range(400):
    p = sigmoid(X_std @ w + b)
    losses.append(bce_loss(y_anem, p))
    err = p - y_anem
    w -= 0.1 * (X_std.T @ err) / len(y_anem)
    b -= 0.1 * float(np.mean(err))

print(f"Initial log-loss: {losses[0]:.5f} -> Final: {losses[-1]:.5f}")
print(f"Learned weights: {np.round(w, 4)} | Intercept b: {b:.4f}")


## Section 23.3.2 — Clinical Confusion Matrix & The Accuracy Paradox

In [ ]:
X_all = clean[feat_cols]
y_stunt = clean["child_stunted"].astype(int)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    X_all, y_stunt, test_size=0.25, stratify=y_stunt, random_state=42
)

stunt_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500)),
])
stunt_pipe.fit(Xs_tr, ys_tr)
ys_pred = stunt_pipe.predict(Xs_te)
ys_prob = stunt_pipe.predict_proba(Xs_te)[:, 1]

tn, fp, fn, tp = confusion_matrix(ys_te, ys_pred).ravel()
acc = (tp + tn) / len(ys_te)
sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
print(f"Confusion Matrix -> TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Accuracy={acc:.4f} | Sensitivity={sens:.4f} | Spec={spec:.4f}")
print(f"Precision (PPV)={prec:.4f} | NPV={npv:.4f}")


## Section 23.3.3 — ROC-AUC vs. PR-AUC on Imbalanced Clinical Outcomes

In [ ]:
roc_st = roc_auc_score(ys_te, ys_prob)
pr_st = average_precision_score(ys_te, ys_prob)
brier_st = brier_score_loss(ys_te, ys_prob)
prev_st = float(ys_te.mean())

print(f"Stunting test prevalence (PR baseline): {prev_st:.4f}")
print(
    f"ROC-AUC: {roc_st:.4f} | PR-AUC: {pr_st:.4f} | "
    f"Brier Score: {brier_st:.4f}"
)


## Section 23.3.4 — Probability Calibration, Platt Scaling & Expected Calibration Error (ECE)

In [ ]:
y_anemia = clean["maternal_anemia"].astype(int)
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_all, y_anemia, test_size=0.25, stratify=y_anemia, random_state=42
)
anem_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500)),
])
anem_pipe.fit(Xa_tr, ya_tr)
p_raw = anem_pipe.predict_proba(Xa_te)[:, 1]

anem_cal = CalibratedClassifierCV(anem_pipe, method="sigmoid", cv=5)
anem_cal.fit(Xa_tr, ya_tr)
p_cal = anem_cal.predict_proba(Xa_te)[:, 1]

# Same ranking, badly overconfident probabilities (p**0.35)
p_overconf = np.clip(p_cal ** 0.35, 0.0, 1.0)

def quick_ece(y_true, y_prob, n_bins=5):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (yp >= lo) & (yp < hi) if i < n_bins - 1 else (yp >= lo)
        if np.any(m):
            ece += (np.sum(m) / len(yt)) * abs(yt[m].mean() - yp[m].mean())
    return float(ece)

for name, probs in [
    ("Raw LogReg", p_raw),
    ("Platt Calib", p_cal),
    ("Overconfident", p_overconf),
]:
    print(
        f"{name:13s} | ROC={roc_auc_score(ya_te, probs):.4f} | "
        f"Brier={brier_score_loss(ya_te, probs):.4f} | "
        f"ECE={quick_ece(ya_te, probs):.4f}"
    )


## Section 23.3.5 — Decision Threshold Tuning & Youden's J Statistic

In [ ]:
fpr, tpr, thresholds = roc_curve(ys_te, ys_prob)
j_scores = tpr - fpr
best_idx = int(np.argmax(j_scores))
best_t = float(thresholds[best_idx])
best_j = float(j_scores[best_idx])
print(f"Stunting Youden threshold: {best_t:.4f} (J={best_j:.4f})")

rows = []
for t in [0.10, round(best_t, 4), 0.25, 0.50]:
    pred_t = (ys_prob >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(ys_te, pred_t).ravel()
    sens_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0
    spec_t = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0.0
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0.0
    rows.append({
        "thresh": round(t, 4),
        "sens": round(sens_t, 4),
        "spec": round(spec_t, 4),
        "prec": round(prec_t, 4),
        "flagged_pct": round(100.0 * pred_t.mean(), 2),
    })
print(pd.DataFrame(rows).to_string(index=False))


## Section 23.3.6 — Class Imbalance: `class_weight="balanced"` & Calibration Distortion

In [ ]:
for label, cw in [("default", None), ("balanced", "balanced")]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, class_weight=cw)),
    ])
    model.fit(Xs_tr, ys_tr)
    yp = model.predict(Xs_te)
    yprob = model.predict_proba(Xs_te)[:, 1]
    print(
        f"{label:8s} | acc={(yp == ys_te).mean():.4f} | "
        f"recall={recall_score(ys_te, yp):.4f} | "
        f"PR-AUC={average_precision_score(ys_te, yprob):.4f} | "
        f"Brier={brier_score_loss(ys_te, yprob):.4f}"
    )


## Section 23.3.7 — Break It On Purpose! (Classification Error Anatomy)

In [ ]:
# Intentional Bug #1: Single-class y_true makes ROC-AUC undefined
with warnings.catch_warnings():
    warnings.simplefilter("error", UndefinedMetricWarning)
    roc_auc_score([0, 0, 0, 0], [0.1, 0.4, 0.7, 0.9])


In [ ]:
# Intentional Bug #2: Passing continuous probabilities to confusion_matrix
confusion_matrix([0, 1, 0, 1], [0.12, 0.85, 0.34, 0.67])


In [ ]:
# Intentional Bug #3: Invalid probability > 1.0 passed to brier_score_loss
brier_score_loss([0, 1, 0, 1], [0.2, 1.35, 0.1, 0.8])


## Section 23.4 — Stage 4 Blueprint: End-to-End Calibrated Clinical Triage Pipeline

In [ ]:
def build_clinical_triage_report(raw_df, target="maternal_anemia", seed=42):
    """Train leak-free scaled + Platt-calibrated classifier & audit it."""
    df = raw_df.copy()
    df["residence"] = df["residence"].astype(str).str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    df["wealth_num"] = df["wealth_quintile"].map(w_map)
    cols = ["maternal_age_years", "anc_visits", "wealth_num", "is_urban"]
    sub = df[[target] + cols].dropna()
    X_tr, X_te, y_tr, y_te = train_test_split(
        sub[cols], sub[target].astype(int),
        test_size=0.25, stratify=sub[target], random_state=seed,
    )
    base = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500)),
    ])
    base.fit(X_tr, y_tr)
    p_raw = base.predict_proba(X_te)[:, 1]
    cal = CalibratedClassifierCV(base, method="sigmoid", cv=5)
    cal.fit(X_tr, y_tr)
    p_cal = cal.predict_proba(X_te)[:, 1]
    fpr, tpr, th = roc_curve(y_te, p_cal)
    bi = int(np.argmax(tpr - fpr))
    return {
        "n_test": int(len(y_te)),
        "roc_auc": round(float(roc_auc_score(y_te, p_cal)), 4),
        "pr_auc": round(float(average_precision_score(y_te, p_cal)), 4),
        "brier_raw": round(float(brier_score_loss(y_te, p_raw)), 4),
        "brier_cal": round(float(brier_score_loss(y_te, p_cal)), 4),
        "ece_cal": round(quick_ece(y_te, p_cal, n_bins=5), 4),
        "youden_t": round(float(th[bi]), 4),
    }

for name, col in [
    ("Anemia", "maternal_anemia"),
    ("Stunting", "child_stunted"),
]:
    rep = build_clinical_triage_report(mch_raw, target=col)
    print(
        f"{name:8s} (n={rep['n_test']}) | ROC={rep['roc_auc']:.4f} | "
        f"PR={rep['pr_auc']:.4f} | Youden={rep['youden_t']:.4f}"
    )
    print(
        f"         Brier raw={rep['brier_raw']:.4f} -> "
        f"cal={rep['brier_cal']:.4f} | ECE={rep['ece_cal']:.4f}"
    )
